# PhishCatcher Training Pipeline

This notebook provides an interactive exploration and training workflow for the phishing detection system.

## Modules
- `ml.preprocessing` - Text cleaning
- `ml.feature_engineering` - URL, text, structural features
- `ml.models.classical` - TF-IDF + LR/SVM/XGBoost
- `ml.models.bert_model` - DistilBERT embeddings
- `ml.models.hybrid` - Combined model
- `ml.evaluation` - Metrics & SHAP/LIME

In [ ]:
import sys
sys.path.insert(0, '../app')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

## 1. Load Data

In [ ]:
# Load archive dataset
archive_path = Path('../archive/phishing_email.csv')
df = pd.read_csv(archive_path)
print(f"Total emails: {len(df)}")
print(f"\nClass distribution:")
print(df['label'].value_counts())

In [ ]:
# EDA: Email length distribution
df['text_length'] = df['text_combined'].astype(str).apply(len)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Overall distribution
axes[0].hist(df['text_length'], bins=50, edgecolor='black')
axes[0].set_title('Email Length Distribution')
axes[0].set_xlabel('Text Length')
axes[0].set_ylabel('Count')

# By class
for label, color in [(0, 'green'), (1, 'red')]:
    subset = df[df['label'] == label]
    axes[1].hist(subset['text_length'], bins=30, alpha=0.5, label=f'{"Phishing" if label == 1 else "Legitimate"}', color=color)
axes[1].set_title('Length by Class')
axes[1].legend()
axes[1].set_xlabel('Text Length')

plt.tight_layout()
plt.show()

## 2. Preprocessing

In [ ]:
from ml.preprocessing import EmailPreprocessor

processor = EmailPreprocessor()

# Sample for quick testing
sample_size = 5000
df_sample = df.sample(n=sample_size, random_state=42)

print("Preprocessing emails...")
df_sample['processed_text'] = df_sample['text_combined'].astype(str).apply(processor.preprocess_text)
print("Done!")

## 3. Feature Engineering

In [ ]:
from ml.feature_engineering import FeatureEngineer

engineer = FeatureEngineer()

print("Extracting features...")
features_list = []

for idx, row in df_sample.head(100).iterrows():
    text = row['text_combined']
    # Extract subject and body roughly
    lines = str(text).split('\n')[:10]
    subject = ' '.join(lines[:2])[:200]
    body = ' '.join(lines[2:])[:5000]
    
    features = engineer.extract_all_features(subject, body)
    features_list.append(features)

features_df = pd.DataFrame(features_list)
print(f"Extracted {len(features_list)} samples with {len(features_df.columns)} features")

## 4. Train Classical ML

In [ ]:
# Full training - see train_model.py
# Or run: python train_model.py --data-source archive --sample-size 20000 --model-type full

## 5. Results Comparison

In [ ]:
import json

results_path = Path('../models/model_comparison.json')

if results_path.exists():
    with open(results_path, 'r') as f:
        comparison = json.load(f)
    
    print("MODEL COMPARISON")
    print("="*60)
    
    for model in comparison.get('models', []):
        print(f"\n{model['name']}:")
        for metric, value in model['metrics'].items():
            print(f"  {metric}: {value:.4f}")
    
    print(f"\nBest Model: {comparison.get('best_model')}")
else:
    print("No results found. Run training first:")
    print("python train_model.py --data-source archive --sample-size 20000 --model-type full")

## 6. Quick Inference

In [ ]:
# Test prediction
test_email = {
    'subject': 'URGENT: Your Account Will Be Suspended',
    'body': 'Click here immediately to verify your account. Failure to verify will result in permanent account closure.'
}

# Use the API
# from ml.api import predict_email
# result = predict_email(test_email['subject'], test_email['body'])
# print(result)